In [0]:
%run ../configs/log_configs

In [0]:
# ─────────────────────────────────────────────
# Cell 1: Imports
# ─────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from delta.tables import DeltaTable
from datetime import datetime

today = datetime.now()
year  = today.strftime("%Y")
month = today.strftime("%m")
day   = today.strftime("%d")

In [0]:
# ─────────────────────────────────────────────
# Cell 2: Widgets
# ─────────────────────────────────────────────
dbutils.widgets.text("environment",    "dev")
dbutils.widgets.text("source_name",    "")
dbutils.widgets.text("bronze_table",   "")
dbutils.widgets.text("silver_table",   "")
dbutils.widgets.text("partition_col",  "")
dbutils.widgets.text("z_order_col",    "")
dbutils.widgets.text("scd_type",       "1")
dbutils.widgets.text("primary_key",    "")
dbutils.widgets.text("is_daily",       "Y")
dbutils.widgets.text("notebook_name",  "")
dbutils.widgets.text("pipeline_name",  "")
dbutils.widgets.text("layer",          "silver")

environment   = dbutils.widgets.get("environment")
source_name   = dbutils.widgets.get("source_name")
bronze_table  = dbutils.widgets.get("bronze_table")
silver_table  = dbutils.widgets.get("silver_table")
partition_col = dbutils.widgets.get("partition_col")
z_order_col   = dbutils.widgets.get("z_order_col")
scd_type      = dbutils.widgets.get("scd_type")
primary_key   = dbutils.widgets.get("primary_key")
is_daily      = dbutils.widgets.get("is_daily")
notebook_name = dbutils.widgets.get("notebook_name")
pipeline_name = dbutils.widgets.get("pipeline_name")
layer         = dbutils.widgets.get("layer")

In [0]:
# ─────────────────────────────────────────────
# Cell 3: Type Casting Map
# All bronze cols are StringType
# Silver applies proper types per source
# ─────────────────────────────────────────────

SILVER_CAST_MAP = {

    "transactions": {
        "amount":       "double",
        "txn_datetime": "timestamp"
    },

    "digital_activity": {
        "event_datetime": "timestamp"
    },

    "campaign_interactions": {
        "converted":            "integer",
        "interaction_datetime": "timestamp"
    },

    "customers": {
        "date_of_birth":  "date",
        "onboarded_date": "date"
    },

    "accounts": {
        "value_amount": "double",
        "start_date":   "date",
        "end_date":     "date"
    },

    "product_holdings": {
        "value_amount": "double",
        "start_date":   "date",
        "end_date":     "date"
    },

    "products": {
        "min_income_required": "double"
    },

    "branches":     {},
    "bank_managers": {}
}

In [0]:
# ─────────────────────────────────────────────
# Cell 4: Generic Cleaning + Type Casting
# ─────────────────────────────────────────────

def safe_cast(df, col_name, col_type):
    """
    Uses try_cast for all type conversions.
    Returns NULL for malformed values instead
    of throwing an error — safe for open-ended
    dates, nulls, and dirty data.
    """
    type_map = {
        "double":    "double",
        "integer":   "integer",
        "long":      "long",
        "date":      "date",
        "timestamp": "timestamp",
        "boolean":   "boolean"
    }

    spark_type = type_map.get(col_type)

    if spark_type:
        df = df.withColumn(
            col_name,
            F.expr(
                f"try_cast(`{col_name}` as {spark_type})"
            )
        )
    else:
        df = df.withColumn(
            col_name,
            F.col(f"`{col_name}`").cast(col_type)
        )

    return df


def clean_and_cast(df, source_name, primary_key):
    """
    Step 1 — Print actual columns for debug
    Step 2 — Drop bronze audit columns
    Step 3 — Drop rows with null primary key
    Step 4 — Deduplicate on primary key
    Step 5 — Trim all string columns
    Step 6 — Nullify empty strings
    Step 7 — Safe cast to correct types
    """

    # ── Step 1: Debug — print actual cols ──
    print(f"Actual columns : {df.columns}")
    print(f"Actual dtypes  : {df.dtypes}")

    # ── Step 2: Drop bronze audit cols ─────
    audit_cols = [
        "bib_ingested_at",
        "bib_source",
        "bib_environment",
        "bib_is_deleted"
    ]
    df = df.drop(*[
        c for c in audit_cols
        if c in df.columns
    ])

    # ── Step 3: Drop null primary keys ─────
    before = df.count()
    df = df.filter(
        F.col(primary_key).isNotNull()
    )
    after = df.count()
    if before != after:
        print(
            f"Dropped {before - after} rows "
            f"with null {primary_key}"
        )

    # ── Step 4: Deduplicate on primary key ─
    df = df.dropDuplicates([primary_key])
    print(
        f"After dedup: {df.count()} records"
    )

    # ── Step 5: Trim string columns ────────
    str_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, StringType)
    ]
    for col in str_cols:
        df = df.withColumn(
            col, F.trim(F.col(col))
        )

    # ── Step 6: Nullify empty strings ──────
    for col in str_cols:
        df = df.withColumn(
            col,
            F.when(
                F.col(col) == "", None
            ).otherwise(F.col(col))
        )

    # ── Step 7: Safe cast types ────────────
    cast_map = SILVER_CAST_MAP.get(
        source_name, {}
    )

    for col_name, col_type in cast_map.items():

        if col_name not in df.columns:
            print(
                f"Column '{col_name}' not found "
                f"— skipping cast"
            )
            continue

        try:
            df = safe_cast(df, col_name, col_type)
            print(
                f"Cast: {col_name} → {col_type}"
            )
        except Exception as cast_err:
            print(
                f"Cast failed: {col_name} "
                f"→ {col_type} | {str(cast_err)}"
            )

    return df

In [0]:
# ─────────────────────────────────────────────
# Cell 5: Shared Delta Writer Helper
# ─────────────────────────────────────────────

def _write_delta(df, table_name, adls_path,
                 partition_col):

    is_valid_partition = (
        partition_col
        and partition_col.strip().upper() != "NA"
    )

    # ── Unity Catalog write ────────────────
    uc_writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if is_valid_partition:
        uc_writer = uc_writer.partitionBy(
            partition_col
        )
    uc_writer.saveAsTable(table_name)
    print(f"UC write done      : {table_name}")

    # ── ADLS write ─────────────────────────
    adls_writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if is_valid_partition:
        adls_writer = adls_writer.partitionBy(
            partition_col
        )
    adls_writer.save(adls_path)
    print(f"ADLS write done    : {adls_path}")

In [0]:
# ─────────────────────────────────────────────
# Cell 5: Shared Delta Writer Helper
# ─────────────────────────────────────────────

def _write_delta(df, table_name, adls_path,
                 partition_col):

    # is_valid_partition = (
    #     partition_col
    #     and partition_col.strip().upper() != "NA"
    # )

    # ── Unity Catalog write ────────────────
    uc_writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    # if is_valid_partition:
    #     uc_writer = uc_writer.partitionBy(
    #         partition_col
    #     )
    uc_writer.saveAsTable(table_name)
    print(f"UC write done: {table_name}")

    # ── ADLS write ─────────────────────────
    adls_writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    # if is_valid_partition:
    #     adls_writer = adls_writer.partitionBy(
    #         partition_col
    #     )
    adls_writer.save(adls_path)
    print(f"ADLS write done: {adls_path}")

In [0]:
# ─────────────────────────────────────────────
# Cell 6: Optimize Helper
# ─────────────────────────────────────────────

# def optimize_silver_table(silver_table, z_order_col):

#     is_valid_zorder = (
#         z_order_col
#         and z_order_col.strip().upper() != "NA"
#     )

#     if is_valid_zorder:
#         spark.sql(
#             f"OPTIMIZE {silver_table} "
#             f"ZORDER BY ({z_order_col})"
#         )
#     else:
#         spark.sql(f"OPTIMIZE {silver_table}")

#     spark.sql(
#         f"VACUUM {silver_table} RETAIN 168 HOURS"
#     )

#     print(f"OPTIMIZE + VACUUM done: {silver_table}")

In [0]:
# ─────────────────────────────────────────────
# Cell 7: SCD Type 1 — Upsert
# Used for: transactions, digital_activity,
#           campaign_interactions, products,
#           branches, bank_managers
# ─────────────────────────────────────────────

def apply_scd_type1(df_new, silver_table,
                    primary_key, partition_col,
                    z_order_col, source_name,
                    environment, layer):

    parts  = silver_table.split(".")
    cat    = parts[0]
    schema = parts[1]
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS "
        f"{cat}.{schema}"
    )

    adls_silver_path = get_adls_path(
        environment, layer, f"{source_name}/"
    )

    # ── Silver audit columns ───────────────
    df_new = (
        df_new
        .withColumn(
            "silver_updated_at",
            F.current_timestamp()
        )
        .withColumn(
            "silver_environment",
            F.lit(environment)
        )
    )

    table_exists = spark.catalog.tableExists(
        silver_table
    )

    if not table_exists:
        print(f"First load → {silver_table}")
        _write_delta(
            df_new, silver_table,
            adls_silver_path, partition_col
        )

    else:
        print(f"Merging → {silver_table}")
        delta_table = DeltaTable.forName(
            spark, silver_table
        )

        update_cols = {
            c: f"source.{c}"
            for c in df_new.columns
            if c != primary_key
        }

        (
            delta_table.alias("target")
            .merge(
                df_new.alias("source"),
                f"target.{primary_key} = "
                f"source.{primary_key}"
            )
            .whenMatchedUpdate(set=update_cols)
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"Merge done         : {silver_table}")

        # ── Sync ADLS ─────────────────────
        (
            DeltaTable.forName(spark, silver_table)
            .toDF()
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(adls_silver_path)
        )
        print(f"ADLS sync done     : {adls_silver_path}")

    # optimize_silver_table(silver_table, z_order_col)

    return (
        DeltaTable.forName(spark, silver_table)
        .toDF()
        .count()
    )

In [0]:
# ─────────────────────────────────────────────
# Cell 8: SCD Type 2 — Track History
# Used for: customers, accounts, product_holdings
# ─────────────────────────────────────────────

def apply_scd_type2(df_new, silver_table,
                    primary_key, partition_col,
                    z_order_col, source_name,
                    environment, layer):

    parts  = silver_table.split(".")
    cat    = parts[0]
    schema = parts[1]
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS "
        f"{cat}.{schema}"
    )

    adls_silver_path = get_adls_path(
        environment, layer, f"{source_name}/"
    )

    now = F.current_timestamp()

    # ── SCD2 tracking columns ──────────────
    df_new = (
        df_new
        .withColumn("scd_start_date", now)
        .withColumn(
            "scd_end_date",
            F.lit(None).cast("timestamp")
        )
        .withColumn(
            "scd_is_current",
            F.lit(True)
        )
        .withColumn(
            "scd_version",
            F.lit(1).cast("integer")
        )
        .withColumn(
            "silver_environment",
            F.lit(environment)
        )
    )

    table_exists = spark.catalog.tableExists(
        silver_table
    )

    if not table_exists:
        print(f"First load SCD2 → {silver_table}")
        _write_delta(
            df_new, silver_table,
            adls_silver_path, partition_col
        )

    else:
        print(f"SCD2 merge → {silver_table}")
        delta_table = DeltaTable.forName(
            spark, silver_table
        )

        # ── Cols to detect changes ─────────
        scd_system_cols = {
            primary_key,
            "scd_start_date",
            "scd_end_date",
            "scd_is_current",
            "scd_version",
            "silver_environment"
        }

        compare_cols = [
            c for c in df_new.columns
            if c not in scd_system_cols
        ]

        change_condition = " OR ".join([
            f"target.{c} <> source.{c}"
            for c in compare_cols
        ])

        # ── Step 1: Expire changed rows ────
        (
            delta_table.alias("target")
            .merge(
                df_new.alias("source"),
                f"target.{primary_key} = "
                f"source.{primary_key} "
                f"AND target.scd_is_current = true"
            )
            .whenMatchedUpdate(
                condition=change_condition,
                set={
                    "scd_end_date":
                        "source.scd_start_date",
                    "scd_is_current": "false"
                }
            )
            .execute()
        )
        print("Expired changed records")

        # ── Step 2: Insert new/changed ─────
        active_pks = (
            DeltaTable.forName(spark, silver_table)
            .toDF()
            .filter(
                F.col("scd_is_current") == True
            )
            .select(primary_key)
        )

        df_to_insert = df_new.join(
            active_pks,
            on=primary_key,
            how="left_anti"
        )

        insert_count = df_to_insert.count()
        print(f"Inserting {insert_count} rows")

        if insert_count > 0:
            (
                df_to_insert.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(silver_table)
            )

        # ── Step 3: Sync ADLS ──────────────
        (
            DeltaTable.forName(spark, silver_table)
            .toDF()
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(adls_silver_path)
        )
        print(f"ADLS sync done     : {adls_silver_path}")

    # optimize_silver_table(silver_table, z_order_col)

    return (
        DeltaTable.forName(spark, silver_table)
        .toDF()
        .count()
    )

In [0]:
# ─────────────────────────────────────────────
# Cell 9: Execute
# ─────────────────────────────────────────────
try:
    # ── 1. Read from Bronze ────────────────
    print(f"Reading bronze     : {bronze_table}")
    df_bronze = spark.read.table(bronze_table)

    # ── 2. Filter by ingestion date ────────
    if is_daily == "Y":
        df_bronze = df_bronze.filter(
            F.to_date(F.col("bib_ingested_at"))
            == F.current_date()
        )
        print(
            f"Filtered to today's ingestion"
        )

    # ── 3. Cache bronze ────────────────────
    df_bronze.cache()
    bronze_count = df_bronze.count()
    print(
        f"Bronze records     : {bronze_count}"
    )

    if bronze_count == 0:
        print(
            f"No records found in bronze "
            f"for {source_name} — skipping"
        )
        log_pipeline_event(
            notebook_name=notebook_name,
            pipeline_name=pipeline_name,
            source_name=source_name,
            layer=layer,
            status="SKIPPED",
            records_processed=0,
            message=(
                f"No bronze records found "
                f"for {source_name}"
            ),
            environment=environment
        )

    else:
        # ── 4. Clean + cast ────────────────
        df_clean = clean_and_cast(
            df_bronze,
            source_name,
            primary_key
        )
        print(
            f"Clean complete     : "
            f"{df_clean.count()} records"
        )

        df_bronze.unpersist()

        # ── 5. Apply SCD ───────────────────
        if scd_type == "2":
            print("Applying SCD Type 2")
            records = apply_scd_type2(
                df_clean, silver_table,
                primary_key, partition_col,
                z_order_col, source_name,
                environment, layer
            )
        else:
            print("Applying SCD Type 1")
            records = apply_scd_type1(
                df_clean, silver_table,
                primary_key, partition_col,
                z_order_col, source_name,
                environment, layer
            )

        # ── 6. Log SUCCESS ─────────────────
        log_pipeline_event(
            notebook_name=notebook_name,
            pipeline_name=pipeline_name,
            source_name=source_name,
            layer=layer,
            status="SUCCESS",
            records_processed=records,
            message=(
                f"{source_name} silver complete. "
                f"SCD{scd_type}. "
                f"Table: {silver_table}"
            ),
            environment=environment
        )
        print(
            f"\nSUCCESS: {source_name} → "
            f"{silver_table} | "
            f"{records} total records"
        )

except Exception as e:
    log_pipeline_event(
        notebook_name=notebook_name,
        pipeline_name=pipeline_name,
        source_name=source_name,
        layer=layer,
        status="FAILED",
        records_processed=0,
        message=f"Silver load failed: {str(e)}",
        environment=environment
    )
    raise

In [0]:
# %sql
# SELECT COUNT(*), TO_DATE(bib_ingested_at) as ingested_date
# FROM bib_catalog.bronze.transactions
# GROUP BY TO_DATE(bib_ingested_at)
# ORDER BY ingested_date DESC